# OpenRouter sample annotation with Leiden clusters

This tutorial shows how to annotate one or more samples with `CellAnnotator`
using an OpenRouter model and a user-provided Leiden key.

In [ ]:
import scanpy as sc
from cell_annotator import CellAnnotator

## Configuration

- `OPENROUTER_API_KEY`: your OpenRouter API key
- `OPENROUTER_MODEL`: model slug (e.g. `openai/gpt-4o-mini`)
- `LEIDEN_KEY`: cluster column in `adata.obs`
- `SAMPLE_KEY`: sample column in `adata.obs`, or `None` for a single sample
- `ADATA_PATH`: path to your `.h5ad` dataset

In [ ]:
OPENROUTER_API_KEY = ""  # e.g. sk-or-v1-...
OPENROUTER_MODEL = "openai/gpt-4o-mini"
LEIDEN_KEY = "leiden"
ADATA_PATH = "path/to/your_data.h5ad"

SPECIES = "human"
TISSUE = "pancreas"
STAGE = "adult"
SAMPLE_KEY = "sample"  # set to None for single-sample datasets

if not OPENROUTER_API_KEY:
    raise ValueError("Set OPENROUTER_API_KEY before continuing.")
if not OPENROUTER_MODEL:
    raise ValueError("Set OPENROUTER_MODEL before continuing.")

## Load data

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)

if LEIDEN_KEY not in adata.obs.columns:
    raise KeyError(f"Column '{LEIDEN_KEY}' was not found in adata.obs")
if SAMPLE_KEY is not None and SAMPLE_KEY not in adata.obs.columns:
    raise KeyError(f"Column '{SAMPLE_KEY}' was not found in adata.obs")

print(adata)
print("Leiden key:", LEIDEN_KEY)
print("Sample key:", SAMPLE_KEY)

## Initialize annotator

In [ ]:
cell_ann = CellAnnotator(
    adata=adata,
    species=SPECIES,
    tissue=TISSUE,
    stage=STAGE,
    cluster_key=LEIDEN_KEY,
    sample_key=SAMPLE_KEY,
    provider="openrouter",
    model=OPENROUTER_MODEL,
    api_key=OPENROUTER_API_KEY,
)

cell_ann

## Run annotation

In [ ]:
cell_ann.get_expected_cell_type_markers(n_markers=3)
cell_ann.get_cluster_markers()
cell_ann.annotate_clusters(key_added="cell_type_predicted")

## Inspect and save results

In [ ]:
adata.obs[[LEIDEN_KEY, "cell_type_predicted"]].head(10)

In [ ]:
if "X_umap" in adata.obsm:
    sc.pl.umap(adata, color=[LEIDEN_KEY, "cell_type_predicted"], wspace=0.35)
else:
    print("No UMAP embedding found; skipping plot.")

output_path = ADATA_PATH.replace(".h5ad", "_annotated.h5ad")
adata.write(output_path)
print(f"Saved annotated object to: {output_path}")